# IDVerify: Face Recognition with ArcFace + ExpFace

**arXiv ref:** *ExpFace: Exponential Angular Margin Loss* — arxiv:2509.19753, 2025

**NPTFace:** *Native Pose-aligned Transformer for Face Recognition* — CVPRW 2026

This notebook demonstrates face embedding extraction and verification using **ArcFace** with an additive angular margin loss.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_dataloaders, make_verification_pairs
from src.model import FaceEmbedder
from src.core import compute_verification_metrics
from src.visualizations import plot_face_sample, plot_similarity_histogram


In [ ]:
# Generate synthetic face data
data = make_synthetic(num_identities=10, imgs_per_id=20, seed=42)
print(f"Faces: {data['n_samples']}, Identities: {data['num_identities']}")
print(f"Image shape: {data['images'].shape}")


# Show sample faces
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = (data['images'][i].permute(1,2,0).numpy() * 127.5 + 127.5).astype(np.uint8)
    ax.imshow(img); ax.axis('off'); ax.set_title(f'ID {data["labels"][i].item()}')
plt.tight_layout(); plt.show()


In [ ]:
# Build ArcFace model
model = FaceEmbedder(embedding_dim=128, num_classes=data['num_identities'], margin=0.5, scale=64.0)
print(f"Model: ResNet-50 backbone + ArcFace head ({sum(p.numel() for p in model.parameters()):,} params)")
print("ArcFace loss: L = -log(exp(s·cos(θ_y+m)) / (exp(s·cos(θ_y+m)) + Σ exp(s·cos(θ_j))))")


In [ ]:
# Extract embeddings
model.eval()
with torch.no_grad():
    embeddings = model(data['images'])
print(f"Embeddings: {embeddings.shape} (unit-norm, {embeddings.norm(dim=1).mean():.4f} avg norm)")


In [ ]:
# Verification demo
pairs, targets = make_verification_pairs(data, num_pairs=100, seed=99)
metrics = compute_verification_metrics(model, pairs, targets)
print(f"ROC AUC: {metrics['roc_auc']:.4f}")
print(f"Best threshold: {metrics['best_threshold']:.3f}")
print(f"F1 Score: {metrics['f1']:.4f}")


In [ ]:
print('IDVerify notebook complete. ArcFace embeddings lie on a hypersphere with angular margin separation.')
